# 02. Model Test — Backbone Load & Forward Pass
# 02. 모델 테스트 — Backbone 로드 및 Forward Pass 확인

**Goal / 목표**
- Load EfficientNet-B0 / ResNet-50 backbone and verify forward pass with a single image
- EfficientNet-B0 / ResNet-50 backbone을 로드하고 단일 이미지로 forward pass를 확인한다

**Execution Order / 실행 순서**
1. Import libraries / 라이브러리 임포트
2. Path & settings / 경로 및 설정
3. Load backbone / Backbone 로드
4. Load a single image / 단일 이미지 로드
5. Forward pass / Forward pass 실행
6. Verify output shape / 출력 형태 확인

---
## 1. Import Libraries / 라이브러리 임포트

In [1]:
import numpy as np
import cv2
import torch
import torch.nn as nn
from torchvision import models, transforms
from pathlib import Path

# 디바이스 설정 / Device setup
device = torch.device(
    'cuda' if torch.cuda.is_available() else
    'mps'  if torch.backends.mps.is_available() else
    'cpu'
)

print('Libraries loaded / 라이브러리 로드 완료')
print(f'PyTorch version / 버전: {torch.__version__}')
print(f'Device / 디바이스: {device}')

Libraries loaded / 라이브러리 로드 완료
PyTorch version / 버전: 2.11.0
Device / 디바이스: mps


---
## 2. Path & Settings / 경로 및 설정

> config.yaml 없이 직접 경로와 값을 지정한다.
> Paths and values are set directly without config.yaml.

In [2]:
ROOT_DIR      = Path('../..').resolve()            # CMYK_MAIN/
RAW_DIR       = ROOT_DIR / 'data_set' / 'raw'   # 원본 스캔 이미지 / Raw scan images
LABELED_DIR   = ROOT_DIR / 'data_set' / 'labeled'  # 라벨링 폴더 / Labeled folder

IMAGE_SIZE    = 128          #  기준 크기 /  standard size
NUM_LEVELS    = 6            # Level 0 ~ 5
BACKBONE_NAME = 'efficientnet_b0'  # 'efficientnet_b0' | 'resnet50'

# 고정 ROI 좌표 (x, y, w, h) / Fixed ROI coordinates
ROI_COORDS = {
    'Y': [0,   0,   800, 200],
    'M': [0,   200, 800, 200],
    'C': [0,   400, 800, 200],
    'K': [0,   600, 800, 200],
}

print(f'Backbone / 백본: {BACKBONE_NAME}')
print(f'Image size / 이미지 크기: {IMAGE_SIZE}')
print(f'Raw dir / 원본 경로: {RAW_DIR}')

Backbone / 백본: efficientnet_b0
Image size / 이미지 크기: 128
Raw dir / 원본 경로: /Users/yangjin-hyeong/Desktop/CMYK_MAIN/data_set/raw


---
## 3. Load Backbone / Backbone 로드

- Head (Classifier) is removed — only the feature extractor part is used
- Head (Classifier) 를 제거하고 특징 추출 부분만 사용한다
- EfficientNet-B0: feature_dim = 1280
- ResNet-50: feature_dim = 2048

In [3]:
if BACKBONE_NAME == 'efficientnet_b0':
    from torchvision.models import EfficientNet_B0_Weights
    backbone    = models.efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
    feature_dim = backbone.classifier[1].in_features  # 1280
    backbone.classifier = nn.Identity()               # Head 제거 / Remove head

elif BACKBONE_NAME == 'resnet50':
    from torchvision.models import ResNet50_Weights
    backbone    = models.resnet50(weights=ResNet50_Weights.DEFAULT)
    feature_dim = backbone.fc.in_features              # 2048
    backbone.fc = nn.Identity()                        # Head 제거 / Remove head

else:
    raise ValueError(f'Unsupported backbone / 지원하지 않는 backbone: {BACKBONE_NAME}')

backbone = backbone.to(device)
backbone.eval()  # 추론 모드 — BN, Dropout 비활성화 / Inference mode — deactivate BN, Dropout

print(f'Backbone loaded / 로드 완료: {BACKBONE_NAME}')
print(f'Feature dimension / 특징 차원: {feature_dim}')

Backbone loaded / 로드 완료: efficientnet_b0
Feature dimension / 특징 차원: 1280


---
## 4. Load a Single Image / 단일 이미지 로드

- Load one image from data/labeled/ and preprocess it
- data/labeled/ 에서 이미지 1장을 로드하고 전처리를 적용한다
- If no image found, use dummy input
- 이미지가 없으면 더미 입력을 사용한다

In [4]:
# data/labeled/ 에서 첫 번째 이미지 탐색 / Find first image in data_set/labeled/
exts       = {'.png', '.jpg', '.jpeg', '.tiff', '.tif'}
all_images = [p for p in LABELED_DIR.rglob('*') if p.suffix.lower() in exts]

if not all_images:
    print('[WARN] No images found / 이미지 없음')
    print(f'       Path checked / 확인 경로: {LABELED_DIR}')
    print('       Using dummy input instead / 더미 입력 사용')
    USE_DUMMY = True
else:
    USE_DUMMY     = False
    sample_path   = all_images[0]
    print(f'Image found / 이미지 발견: {sample_path}')

    # 이미지 로드 / Load image
    img = cv2.imread(str(sample_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    #  기준 전처리 /  standard preprocessing
    img = cv2.resize(img, (IMAGE_SIZE, IMAGE_SIZE))
    img = img.astype(np.float32) / 255.0   # [0,1] 정규화 / Normalize to [0,1]

    print(f'Image shape / 이미지 형태: {img.shape}')
    print(f'Value range / 값 범위: [{img.min():.3f}, {img.max():.3f}]')

Image found / 이미지 발견: /Users/yangjin-hyeong/Desktop/CMYK_MAIN/data_set/labeled/M/0/scan_024_M_0011.png
Image shape / 이미지 형태: (128, 128, 3)
Value range / 값 범위: [0.498, 1.000]


---
## 5. Forward Pass

- Convert the image to a tensor and run a forward pass through the backbone
- 이미지를 텐서로 변환하고 backbone으로 forward pass를 실행한다

In [5]:
if USE_DUMMY:
    # 더미 입력 생성 / Create dummy input
    tensor = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE).to(device)
    print('[INFO] Using dummy input / 더미 입력 사용 중')
else:
    # (H, W, 3) → (1, 3, H, W) 텐서 변환 / Convert to tensor
    tensor = torch.tensor(img).permute(2, 0, 1).unsqueeze(0).to(device)

print(f'Input tensor shape / 입력 텐서 형태: {tensor.shape}')

# Forward pass 실행 / Run forward pass
with torch.no_grad():
    features = backbone(tensor)  # (1, feature_dim)

print(f'Output feature shape / 출력 특징 형태: {features.shape}')

Input tensor shape / 입력 텐서 형태: torch.Size([1, 3, 128, 128])
Output feature shape / 출력 특징 형태: torch.Size([1, 1280])


---
## 6. Verify Output Shape / 출력 형태 확인

In [6]:
expected_shape = (1, feature_dim)

print('=' * 50)
print('  Forward Pass Verification / 검증 결과')
print('=' * 50)
print(f'  Backbone      : {BACKBONE_NAME}')
print(f'  Input shape   : {tuple(tensor.shape)}')
print(f'  Output shape  : {tuple(features.shape)}')
print(f'  Expected shape: {expected_shape}')
print(f'  Device        : {device}')
print('=' * 50)

if tuple(features.shape) == expected_shape:
    print('  [PASS] Output shape is correct / 출력 형태 정상')
else:
    print('  [FAIL] Output shape mismatch / 출력 형태 불일치')
    print(f'         Got {tuple(features.shape)}, expected {expected_shape}')

  Forward Pass Verification / 검증 결과
  Backbone      : efficientnet_b0
  Input shape   : (1, 3, 128, 128)
  Output shape  : (1, 1280)
  Expected shape: (1, 1280)
  Device        : mps
  [PASS] Output shape is correct / 출력 형태 정상


---
## 7. (Optional) Switch Backbone / (선택) Backbone 교체 테스트

- Change BACKBONE_NAME in Cell 2 and re-run cells 3~6 to compare
- 셀 2의 BACKBONE_NAME 을 변경하고 3~6번 셀을 다시 실행하여 비교한다

| Backbone | Feature Dim | Model Size |
|---|---|---|
| efficientnet_b0 | 1280 | ~5.3 MB |
| resnet50 | 2048 | ~98 MB |